## Human data curation

Multi-modal xarray serialized as NetCDF (.nc) files, created from the matlab pre-processed (*.mat)

In [ ]:
from pathlib import Path
import os

import numpy as np
import xarray as xr

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()

from core import config, data

tau_vol7_folder = '/data/healthy_volunteers/tau_vol7_feb19_2024/eval/'
tau_vol8_folder = '/data/healthy_volunteers/tau_vol8_mar_2024/eval'
tau_vol9_folder = '/data/healthy_volunteers/tau_vol9_apr_2024/eval'
tau_vol10_folder = '/data/healthy_volunteers/tau_vol10_apr_2024/eval'

config.Config.configure_AMIDE_clinical_whole_brain()

In [ ]:
def brainds2xarray(brainds, data):    
    mrf_coords = np.arange(31)
    x_coords = np.arange(brainds.shape[0])*1.8
    z_coords = np.arange(brainds.shape[1])*1.8
    y_coords = np.arange(brainds.shape[2])*1.8
    brainxarray = xr.Dataset(
        {
            #'measured_normed_T': (['time', 'coronal', 'axial', 'saggital'], brainds.measured_normed_T),
            'MT_data': (['time', 'coronal', 'axial', 'saggital'], data.MT_data),
            'AMIDE_data': (['time', 'coronal', 'axial', 'saggital'], data.AMIDE_data),
            'roi_mask_nans': (['coronal', 'axial', 'saggital'], brainds.roi_mask_nans),
            'B1_fix_factor_map': (['coronal', 'axial', 'saggital'], brainds.B1_fix_factor_map),
            'B0_shift_ppm_map': (['coronal', 'axial', 'saggital'], brainds.B0_shift_ppm_map),
            'T2ms': (['coronal', 'axial', 'saggital'], 1000/brainds.R2a_V),
            'T1ms': (['coronal', 'axial', 'saggital'], 1000/brainds.R1a_V), 
            'white_mask': (['coronal', 'axial', 'saggital'], data.white), 
            'gray_mask': (['coronal', 'axial', 'saggital'], data.gray),        
        },
        coords={
            'MRF_cycles': ('MRF_cycles', mrf_coords, {'units': 'cycle'}),        
            'coronal': ('coronal', x_coords, {'units': 'mm'}), 
            'axial': ('axial', z_coords, {'units': 'mm'}),        
            'saggital': ('saggital', y_coords, {'units': 'mm'}),   
        }
    )

    return brainxarray

cb = 50  # def 45

vol7_xa = brainds2xarray(
    data.SlicesFeed.from_cestmrf_folder(tau_vol7_folder, mt_or_amide='mt', crop_bot=cb),
    data.Folder2Data().load_erl(tau_vol7_folder, crop_bot=cb, raw_folder='', drop_first=False)
)
vol7_xa.to_netcdf('data/vol7.nc')

vol8_xa = brainds2xarray(
    data.SlicesFeed.from_cestmrf_folder(tau_vol8_folder, mt_or_amide='mt', crop_bot=cb),
    data.Folder2Data().load_erl(tau_vol8_folder, crop_bot=cb, raw_folder='', drop_first=False)
)
vol8_xa.to_netcdf('data/vol8.nc')

vol9_xa = brainds2xarray(
    data.SlicesFeed.from_cestmrf_folder(tau_vol9_folder, mt_or_amide='mt', crop_bot=cb),
    data.Folder2Data().load_erl(tau_vol9_folder, crop_bot=cb, raw_folder='', drop_first=False)
)
vol9_xa.to_netcdf('data/vol9.nc')

vol10_xa = brainds2xarray(
    data.SlicesFeed.from_cestmrf_folder(tau_vol10_folder, mt_or_amide='mt', crop_bot=cb),
    data.Folder2Data().load_erl(tau_vol10_folder, crop_bot=cb, raw_folder='', drop_first=False)
)
vol10_xa.to_netcdf('data/vol10.nc')

xr.concat(
    [vol7_xa, vol8_xa, vol9_xa, vol10_xa],
    dim='axial',
    coords='all',
    compat='override'
).to_netcdf('data/vol7-10.nc')

xr.concat(
    [vol7_xa, vol9_xa, vol10_xa],
    dim='axial',
    coords='all',
    compat='override'
).to_netcdf('data/vol7-9-10.nc')

xr.concat(
    [vol7_xa, vol8_xa, vol10_xa],
    dim='axial',
    coords='all',
    compat='override'
).to_netcdf('data/vol7-8-10.nc')

xr.concat(
    [vol7_xa, vol8_xa, vol9_xa],
    dim='axial',
    coords='all',
    compat='override'
).to_netcdf('data/vol7-8-9.nc')

xr.concat(
    [vol8_xa, vol9_xa, vol10_xa],
    dim='axial',
    coords='all',
    compat='override'
).to_netcdf('data/vol8-9-10.nc')

In [ ]:
xa_7_10 = xr.open_dataset('data/vol7-10.nc')
xa_7_10_coarse = xa_7_10.coarsen(axial=3, boundary='trim').mean()
xa_7_10_coarse.to_netcdf('data/vol7-10_coarse.nc')

## Mice data curation

### loading from Ron's repos

In [ ]:
import shutil, glob, os, numpy as np
from pathlib import Path

from helpers import load_mouse_dicoms as lmd

# Example: Copy files with glob pattern
src_pattern = '/home/alexf/codez/mice_nbmf/Rons/January_results/Control_None_2025-01-09/*.npy'
dest_dir = '/home/alexf/codez/pediatric-tumor-mice/Pediatric Brain Tumors-Jan 2025/20250109_101440_OrPerlman_OrZohar_C1_None_1__1_1_1/'

data_folders = ['/home/alexf/codez/pediatric-tumor-mice/Pediatric Brain Tumors-Jan 2025/' + x for x in [
    '20250109_101440_OrPerlman_OrZohar_C1_None_1__1_1_1',
    '20250109_082312_OrPerlman_OrZohar_C1_2R_1_1_1',  # or is it 20250109_133953_OrPerlman_OrZohar_C2_2R_1_1_1 ?
    '20250109_063215_OrPerlman_OrZohar_C1_2L_1_1_1',
    '20250109_115848_OrPerlman_OrZohar_C1_1L1R_1_1_1'
]]
mask_folders = ['/home/alexf/codez/mice_nbmf/Rons/January_results/' + x for x in [
    'Control_None_2025-01-09', 
    'Control_2R_2025-01-09',
    'Control_2L_2025-01-09',
    'Control_1L1R_2025-01-09'
]]
subfolder_dicts = [
    {'t1': '6', 't2': '13', 'amide': '8', 'mt': '9'},  # None .. 8-9-10-14 mt apt rnoe?
    {'t1': '4', 't2': '11', 'amide': '6', 'mt': '7'},  # 2R
    {'t1': '7', 't2': '14', 'amide': '9', 'mt': '10'},  # 2L
    {'t1': '4', 't2': '11', 'amide': '6', 'mt': '7'}   # 1L1R
]
# !! Found out by running manually the below for each folder - see below
# egrep -e '(amide51|mt52|T1map|T2map)' /home/alexf/codez/pediatric-tumor-mice/Pediatric\ Brain\ Tumors-Jan\ 2025/20250109_115848_OrPerlman_OrZohar_C1_1L1R_1_1_1/*/acqp 

for (masks_folder, data_folder) in zip(mask_folders, data_folders):
    src_pattern = masks_folder+'/*_mask.npy'
    for src_file in glob.glob(src_pattern):
        if os.path.isfile(src_file):  # only copy files, not directories
            shutil.copy(src_file, data_folder)     
            print(f'Copied {src_file}')       
            

In [ ]:
for (masks_folder, data_folder, subfolder_dict) in zip(mask_folders, data_folders, subfolder_dicts):
    !ls {data_folder.replace(' ','\\ ')}/*npy
    combined_ds = lmd.load_and_concat_xarrays(base_path=data_folder, subfolder_dict=subfolder_dict)
    lmd.visualize_dataset(combined_ds)
    brain_mask = combined_ds['brain_mask'].to_numpy().copy()
    brain_mask = np.float32(brain_mask)
    brain_mask[brain_mask==0] = np.nan
    combined_ds['roi_mask_nans'] = (('height', 'slice', 'width'), brain_mask[:, None, :])
    dbname = f'ped_{masks_folder.split('/')[-1]}.nc'
    combined_ds.to_netcdf('data/'+dbname)
    combined_ds.to_netcdf('/data/mice/'+dbname)
    

In [ ]:
import glob
from pathlib import Path
import pandas as pd
import re

# Count files in each data folder matching the pattern
results = []
for folder in data_folders:
    print(folder)
    for subp in Path(folder).iterdir():
        if not subp.is_dir():
            continue
        dicom_dir = subp / 'pdata' / '1' / 'dicom'
        file_count = len(list(dicom_dir.glob('*'))) if dicom_dir.exists() else 0
        if file_count in [6, 25, 32]:            
            acqp_matches = []
            acqp_file = subp / 'acqp'
            if acqp_file.exists():
                acqp_matches = [
                    l.strip()
                    for l in acqp_file.read_text().splitlines()
                    if re.match(r'(amide51|mt52|T1map|T2map)', l) 
                ]
            else:
                assert 0
            print(f'.../{str(subp).split('/')[-1]}, dicoms: {file_count}, proto: {acqp_matches}')    

### ozohar Nov2024 

In [ ]:
base_path = '/data/mice/20240404_125156_Operlman_ozohar_mouse2_right_2_1_2'

# Create binary masks from using PixLabel polygon annotations
# Example workflow:
combined_ds = lmd.load_and_concat_xarrays(base_path, subfolder_dict={'t1': '4', 't2': '5', 'amide': '6', 'mt': '7'})
# 1. Upload t1 image to PixLabel for manual segmentation
# 2. Download resulting JSON with polygon coordinates
# 3. Use create_binary_mask to convert to numpy masks
masks, polygons = lmd.create_binary_mask(
    '/data/mice/20240404_125156_Operlman_ozohar_mouse2_right_2_1_2/ozohar_right_20240404_125156.json',
    (40, 64))
np.save(base_path+'/brain_mask', masks[0])
np.save(base_path+'/tumor_mask', masks[1])
np.save(base_path+'/contralateral_mask', masks[2])


In [ ]:
from helpers import load_mouse_dicoms as lmd

base_path = '/data/mice/20240404_125156_Operlman_ozohar_mouse2_right_2_1_2'
combined_ds = lmd.load_and_concat_xarrays(base_path, subfolder_dict={'t1': '4', 't2': '5', 'amide': '6', 'mt': '7'})
lmd.visualize_dataset(combined_ds)

base_path = '/data/mice/20240404_091158_Operlman_ozohar_mouse3_two_ears_1_1_1/'
combined_ds = lmd.load_and_concat_xarrays(base_path, subfolder_dict={'t1': '7', 't2': '10', 'amide': '11', 'mt': '12'})
lmd.visualize_dataset(combined_ds)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
brain_mask = combined_ds['brain_mask'].to_numpy().copy()
brain_mask = np.float32(brain_mask)
brain_mask[brain_mask==0] = np.nan
combined_ds['roi_mask_nans'] = (('height', 'slice', 'width'), brain_mask[:, None, :])


In [ ]:
combined_ds

## Removed content - see in older branches

### py processed (*_aligned.nii) --> xarray

(!!) Testing a Python-based alternative to the whole MATLAB preprocessing pipeline
 - see in older branches.


### Misc phantom analyses?